In [1]:
# ==========================================================
# Imports
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from collections import Counter

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ==========================================================
# Paths
# ==========================================================

PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

SENSOR_DATASET_PATH = (
    PROJECT_PATH
    / "datasets"
    / "processed"
    / "uah_dataset.npz"
)

VIDEO_FEATURE_PATH = (
    PROJECT_PATH
    / "datasets"
    / "processed"
    / "video_features"
)

In [4]:
sensor_data = np.load(SENSOR_DATASET_PATH)

X = sensor_data["X"]
y = sensor_data["y"]
groups = sensor_data["groups"]

print(X.shape)
print(y.shape)
print(groups.shape)

(30676, 120, 13)
(30676,)
(30676,)


In [5]:
video_feature_files = sorted(
    VIDEO_FEATURE_PATH.glob("*.npy")
)

print(len(video_feature_files))

40


In [6]:
unique_groups = np.unique(groups)

trip_labels = np.array([
    y[groups == trip][0]
    for trip in unique_groups
])

print(unique_groups)
print(trip_labels)

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39]
[0 0 0 2 1 2 1 0 2 1 0 0 2 1 0 1 0 0 2 1 2 0 0 2 1 0 2 1 0 2 1 0 0 2 1 0 1
 0 1 2]


In [7]:
label_names = {
    0: "NORMAL",
    1: "DROWSY",
    2: "AGGRESSIVE",
}

trip_counter = Counter(trip_labels)

for label, count in sorted(trip_counter.items()):
    print(label_names[label], count)

NORMAL 17
DROWSY 12
AGGRESSIVE 11


In [8]:
# ==========================================================
# Trip-level Stratified Split
# ==========================================================

train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=0.20,
    random_state=42,
    stratify=trip_labels,
)

print("Train trips:", len(train_groups))
print("Test trips :", len(test_groups))

print()
print("Train:", train_groups)
print("Test :", test_groups)

Train trips: 32
Test trips : 8

Train: [18 34  7 27  2  5 31 21 24 38 19 39  0 17 35 10 14 26  9 13  1 12 23 33
 30 20 11  4  8 25 36 37]
Test : [28 22  6 15 32 29 16  3]


In [9]:
train_trip_labels = trip_labels[
    np.isin(unique_groups, train_groups)
]

test_trip_labels = trip_labels[
    np.isin(unique_groups, test_groups)
]

print("TRAIN")
print(Counter(train_trip_labels))

print()

print("TEST")
print(Counter(test_trip_labels))

TRAIN
Counter({np.int64(0): 13, np.int64(1): 10, np.int64(2): 9})

TEST
Counter({np.int64(0): 4, np.int64(2): 2, np.int64(1): 2})


In [10]:
train_counter = Counter(train_trip_labels)
test_counter = Counter(test_trip_labels)

df = pd.DataFrame(
    {
        "Train Trips": [
            train_counter[0],
            train_counter[1],
            train_counter[2],
        ],
        "Test Trips": [
            test_counter[0],
            test_counter[1],
            test_counter[2],
        ],
    },
    index=[
        "NORMAL",
        "DROWSY",
        "AGGRESSIVE",
    ],
)

df["Train %"] = (
    df["Train Trips"]
    / df["Train Trips"].sum()
    * 100
).round(2)

df["Test %"] = (
    df["Test Trips"]
    / df["Test Trips"].sum()
    * 100
).round(2)

print(df)

            Train Trips  Test Trips  Train %  Test %
NORMAL               13           4    40.62    50.0
DROWSY               10           2    31.25    25.0
AGGRESSIVE            9           2    28.12    25.0


In [11]:
# ==========================================================
# Build aligned multimodal dataset
# ==========================================================

sensor_samples = []
video_samples = []
labels = []
aligned_groups = []

for trip_id in unique_groups:

    sensor_trip = X[groups == trip_id]

    video_trip = np.load(
        video_feature_files[trip_id]
    )

    label_trip = y[groups == trip_id]

    common_length = min(
        len(sensor_trip),
        len(video_trip),
    )

    sensor_samples.append(
        sensor_trip[:common_length]
    )

    video_samples.append(
        video_trip[:common_length]
    )

    labels.append(
        label_trip[:common_length]
    )

    aligned_groups.append(
        np.full(common_length, trip_id)
    )

In [12]:
sensor_samples = np.concatenate(sensor_samples)
video_samples = np.concatenate(video_samples)
labels = np.concatenate(labels)
aligned_groups = np.concatenate(aligned_groups)

print(sensor_samples.shape)
print(video_samples.shape)
print(labels.shape)
print(aligned_groups.shape)

(30568, 120, 13)
(30568, 2048)
(30568,)
(30568,)


In [13]:
train_mask = np.isin(
    aligned_groups,
    train_groups
)

test_mask = np.isin(
    aligned_groups,
    test_groups
)

In [14]:
X_sensor_train = sensor_samples[train_mask]
X_sensor_test = sensor_samples[test_mask]

X_video_train = video_samples[train_mask]
X_video_test = video_samples[test_mask]

y_train = labels[train_mask]
y_test = labels[test_mask]

In [15]:
print(X_sensor_train.shape)
print(X_video_train.shape)
print(y_train.shape)

print()

print(X_sensor_test.shape)
print(X_video_test.shape)
print(y_test.shape)

(24617, 120, 13)
(24617, 2048)
(24617,)

(5951, 120, 13)
(5951, 2048)
(5951,)


In [16]:
import torch
from torch.utils.data import Dataset


class MultimodalDataset(Dataset):

    def __init__(
        self,
        sensor_data,
        video_data,
        labels,
    ):

        self.sensor = torch.FloatTensor(sensor_data)
        self.video = torch.FloatTensor(video_data)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return (
            self.sensor[idx],
            self.video[idx],
            self.labels[idx],
        )

In [17]:
train_dataset = MultimodalDataset(
    X_sensor_train,
    X_video_train,
    y_train,
)

test_dataset = MultimodalDataset(
    X_sensor_test,
    X_video_test,
    y_test,
)

print(len(train_dataset))
print(len(test_dataset))

24617
5951


In [18]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
)

print(len(train_loader))
print(len(test_loader))

770
186


In [19]:
sensor, video, label = next(iter(train_loader))

print(sensor.shape)
print(video.shape)
print(label.shape)

torch.Size([32, 120, 13])
torch.Size([32, 2048])
torch.Size([32])
